In [ ]:
import numpy as np

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)

def forward_attention(X, Wq, Wk, Wv):
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv
    d_k = Wq.shape[1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    A = softmax(scores)
    out = A @ V
    cache = (X, Q, K, V, A, d_k)
    return out, cache

def backward_attention(dOut, cache, Wq, Wk, Wv):
    """Manual backward pass for single-head self-attention.
    Computes dL/dWq, dL/dWk, dL/dWv without autograd."""
    X, Q, K, V, A, d_k = cache
    n = X.shape[0]

    # 1) grad wrt V and A from out = A @ V
    dV = A.T @ dOut
    dA = dOut @ V.T

    # 2) grad of softmax: dScores_ij = A_ij * (dA_ij - sum_k A_ik * dA_ik)
    row_dot = np.sum(A * dA, axis=-1, keepdims=True)
    dScores = A * (dA - row_dot)
    dScores /= np.sqrt(d_k)

    # 3) scores = Q @ K.T  ->  grads for Q and K
    dQ = dScores @ K
    dK = dScores.T @ Q

    # 4) chain back through the linear projections Q = X@Wq, K = X@Wk, V = X@Wv
    dWq = X.T @ dQ
    dWk = X.T @ dK
    dWv = X.T @ dV

    return dWq, dWk, dWv

if __name__ == "__main__":
    np.random.seed(0)
    seq_lens = [4, 8, 16, 32]
    d_model, d_k = 8, 8

    for n in seq_lens:
        X = np.random.randn(n, d_model)
        Wq = np.random.randn(d_model, d_k) * 0.1
        Wk = np.random.randn(d_model, d_k) * 0.1
        Wv = np.random.randn(d_model, d_k) * 0.1

        out, cache = forward_attention(X, Wq, Wk, Wv)
        dOut = np.random.randn(*out.shape)   # pretend upstream gradient (dLoss/dOut)

        dWq, dWk, dWv = backward_attention(dOut, cache, Wq, Wk, Wv)

        print(f"seq_len={n:3d}  |grad Wq|={np.linalg.norm(dWq):.4f}  "
              f"|grad Wk|={np.linalg.norm(dWk):.4f}  |grad Wv|={np.linalg.norm(dWv):.4f}")

---
## Task 5: Custom Transformer Backpropagation & Gradient Tracking